In [1]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads .env from current/project directory

hf_token = os.getenv("HF_TOKEN")
openai_key = os.getenv("OPENAI_API_KEY")
langfuse_public = os.getenv("LANGFUSE_PUBLIC_KEY")
langfuse_secret = os.getenv("LANGFUSE_SECRET_KEY")

In [7]:
from smolagents import CodeAgent, LiteLLMModel, tool, Tool, DuckDuckGoSearchTool

local_model = LiteLLMModel(
    model_id="ollama/qwen2.5:7b",
    api_base="http://localhost:11434",
    api_key="ollama"
)

agent = CodeAgent(tools=[], model=local_model)

In [4]:
# from smolagents import CodeAgent, InferenceClientModel, tool

# Let's pretend we have a function that fetches the highest-rated catering services.
@tool
def catering_service_tool(query: str) -> str:
    """
    This tool returns the highest-rated catering service in Gotham City.

    Args:
        query: A search term for finding catering services.
    """
    # Example list of catering services and their ratings
    services = {
        "Gotham Catering Co.": 4.9,
        "Wayne Manor Catering": 4.8,
        "Gotham City Events": 4.7,
    }

    # Find the highest rated catering service (simulating search query filtering)
    best_service = max(services, key=services.get)

    return best_service


agent = CodeAgent(tools=[catering_service_tool], model=local_model)

# Run the agent to find the best catering service
result = agent.run(
    "Can you give me the name of the highest-rated catering service in Gotham City?"
)

print(result)   # Output: Gotham Catering Co.

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Can you give me the name of the highest-rated catering service in Gotham City?                                  │
│                                                                                                                 │
╰─ LiteLLMModel - ollama/qwen2.5:7b ──────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  result = catering_service_tool(query="Gotham City")                                                              
  print(result)                                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Gotham Catering Co.

Out: None

[Step 1: Duration 17.29 seconds| Input tokens: 2,103 | Output tokens: 66]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Gotham Catering Co.")                                                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Gotham Catering Co.

[Step 2: Duration 3.06 seconds| Input tokens: 4,351 | Output tokens: 132]

Gotham Catering Co.


In [6]:
class SuperheroPartyThemeTool(Tool):
    name = "superhero_party_theme_generator"
    description = """
    This tool suggests creative superhero-themed party ideas based on a category.
    It returns a unique party theme idea."""

    inputs = {
        "category": {
            "type": "string",
            "description": "The type of superhero party (e.g., 'classic heroes', 'villain masquerade', 'futuristic Gotham').",
        }
    }

    output_type = "string"

    def forward(self, category: str):
        themes = {
            "classic heroes": "Justice League Gala: Guests come dressed as their favorite DC heroes with themed cocktails like 'The Kryptonite Punch'.",
            "villain masquerade": "Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.",
            "futuristic Gotham": "Neo-Gotham Night: A cyberpunk-style party inspired by Batman Beyond, with neon decorations and futuristic gadgets."
        }

        return themes.get(category.lower(), "Themed party idea not found. Try 'classic heroes', 'villain masquerade', or 'futuristic Gotham'.")

# Instantiate the tool
party_theme_tool = SuperheroPartyThemeTool()
agent = CodeAgent(tools=[party_theme_tool], model=local_model)

# Run the agent to generate a party theme idea
result = agent.run(
    "What would be a good superhero party idea for a 'villain masquerade' theme?"
)

print(result)  # Output: "Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains."

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What would be a good superhero party idea for a 'villain masquerade' theme?                                     │
│                                                                                                                 │
╰─ LiteLLMModel - ollama/qwen2.5:7b ──────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  theme_idea = superhero_party_theme_generator(category="villain masquerade")                                      
  print(theme_idea)                                                                                                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.

Out: None

[Step 1: Duration 7.69 seconds| Input tokens: 2,138 | Output tokens: 88]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(theme_idea)                                                                                         
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.

[Step 2: Duration 2.95 seconds| Input tokens: 4,464 | Output tokens: 144]

Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.


# Retrieval Agents

In [8]:
# Initialize the search tool
search_tool = DuckDuckGoSearchTool()



agent = CodeAgent(
    model=local_model,
    tools=[search_tool],
)

# Example usage
response = agent.run(
    "Search for luxury superhero-themed party ideas, including decorations, entertainment, and catering."
)
print(response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Search for luxury superhero-themed party ideas, including decorations, entertainment, and catering.             │
│                                                                                                                 │
╰─ LiteLLMModel - ollama/qwen2.5:7b ──────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  search_query = "luxury superhero-themed party decorations entertainment catering"                                
  search_results = web_search(query=search_query)                                                                  
  print(search_results)                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results

[Creative Party Themes That Will Wow Your 
Guests](https://www.anitadee.com/blog/creative-party-themes-that-will-wow-your-guests/)
Superheroesare all the rage these days, and asuperhero-themedpartyis a surefire hit for kids and adults alike. 
...decorations, and lively music ...

[Birthday Party Catering Ideas: Unique Themes for All 
Ages](https://www.articleted.com/article/851037/281750/Birthday-Party-Catering-Ideas--Unique-Themes-for-All-Ages)
CateringSydney can tailor menus to complement popular themes likesuperheroadventures (think mini pizzas 
withsuperherocutouts as toppings!), ...

[Wedding & Party Network - Online wedding, special event 
and](https://www.weddingandpartynetwork.com/blog/tag/themed-parties/)
Break Out Those Flapper Dresses – The best apart about thispartyis everyone gets to dress up! Who doesn ’ t love 
athemedcostumeparty...

[300+ Party Decoration Ideas for Memorable Celebrations 
-](https://www.shinny-party.com/blog/300+-party-decoration-ideas-for-memorable-celebrations-pinterest-2020/)
...partybut struggling to come up with unique and creativedecorationideas? Look no further than Pinterest for 
endless inspiration! With over 300 ideas ...

[Birthday Party Packages at BIR Luxury Homes | Celebrate in 
Style](https://birluxurylanding.com/birthday-party-packages/)
We offer all-inclusive packages with all the frills and flourishes ofthemeddecorationsto gourmetcatering, so you 
can just sit back, relax, and ...

[30 Elegant Christmas Party Themes & Ideas for a 
Stylish](https://www.catchmyparty.com/blog/elegant-christmas-party-themes)
Timeless HolidayLuxuryThese glamorous Christmaspartythemes channel classic seasonal style — rich textures, glowing 
candlelight, and refined ...

[Gala Dinner Event Activities Archives - 
Partyinkers](https://www.partyinkers.com/category/gala-dinner-event-activities/)
In this guide, we’ll not only explore some of the best Dinner and Dance event venues in Singapore but also show you
how Partyinkers’ innovative ...

[Gala dinner themes Archives - Partyinkers](https://www.partyinkers.com/tag/gala-dinner-themes/)
... theme makes it perfect for summerpartiesor ... Tip : Serve tropical-themeddrinks and have a limbo contest to 
keep the beachpartyspirit alive.

[Kids Party Decoration Suppliers - 
PartiesAndCelebrations](https://www.partiesandcelebrations.com.au/directory/category/decorations/kids-party-decorat
ions/)
Ordering is Easy, just browse our Selection ofPartyDecorationsand Supplies and pick aPartyTheme.... 
...partysupplies,themeddecorations...

[Parties | Venue Hire | Theme Parties | Birthday Parties |](https://www.animalmagicfamily.co.uk/party-venue-hire)
Why not go one step further and have us organise athemedpartyfor you? We specialise in planning horror and movie 
scene settings so whether you re ...

Out: None

[Step 1: Duration 17.70 seconds| Input tokens: 2,113 | Output tokens: 74]

...

Error in code parsing:
Your code snippet is invalid, because the regex pattern <code>(.*?)</code> was not found in it.
            Here is your code snippet:
            Thought: The results for Superhero Decorations, Entertainment, and Catering are quite similar, 
indicating that the keyword search returned a lot of overlapping information. To provide more specific and useful 
recommendations, let's extract and highlight key points from the relevant content:

### Superhero Decorations
1. **Superheroes are all the rage these days, and a superhero-themed party is a surefire hit for kids and adults 
alike.**
2. **Think about adding decorations such as balloons, banners, and themed tableware to create a festive 
atmosphere.**
3. **Consider using themed party favors like superhero capes or masks.**

### Superhero Entertainment
1. **A superhero-themed party is a surefire hit for kids and adults alike.**
2. **Entertainment options could include a superhero-themed photo booth or hire a local performer to dress up as a 
superhero.**
3. **Lively music and games can enhance the fun and excitement of the party.**

### Superhero Catering
1. **Catering can tailor menus to complement popular themes like superhero adventures.**
2. **Mini pizzas with superhero cutouts can be a fun and delicious treat.**
3. **Gourmet catering packages can include themed food and drink options to enhance the overall experience.**

Based on the extracted information, here are some specific recommendations:

- **Decorations:** Use balloons, banners, and themed tableware to create a superhero atmosphere.
- **Entertainment:** Consider a superhero-themed photo booth or a performer, and include lively music and games.
- **Catering:** Customize the menu with themed options like mini pizzas with superhero cutouts or gourmet food and 
drinks.

These recommendations should help you organize a fun and engaging superhero-themed party for kids and adults 
alike!</code>
            Make sure to include code with the correct pattern, for instance:
            Thoughts: Your thoughts
            <code>
            # Your python code here
            </code>
Make sure to provide correct code blobs.

[Step 5: Duration 26.40 seconds| Input tokens: 17,339 | Output tokens: 1,518]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 6 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[Step 6: Duration 26.88 seconds]

KeyboardInterrupt: 